# Antipsychotic Prescribing - Geographic Analysis

**Prerequisite:** Run `00_build_database.ipynb` first.

This notebook investigates geographic variation in antipsychotic prescribing across U.S. states using Medicare Part D 2023 data. The central question is whether rural states show elevated antipsychotic prescribing rates, and if so, whether the pattern differs between oral generic antipsychotics and long-acting injectable (LAI) formulations.

**Analytical framework:**
- If the rural clustering is driven by LAIs more than oral generics, it suggests a treatment access pattern: patients in rural areas cannot make frequent clinic visits, so providers use monthly or quarterly injections
- If both show the same rural pattern, it more likely reflects true geographic variation in disease burden
- These explanations are not mutually exclusive

In [2]:
import sqlite3
import pandas as pd
import plotly.express as px

db_path = r"C:\Users\palla\OneDrive\Documents\Coding Projects\CMS\database\cms.db"
conn = sqlite3.connect(db_path)

## Step 1: Define Drug Groups

Antipsychotics are split into three groups for separate geographic analysis:

1. **Oral generics** — high-volume first and second-generation antipsychotics available as daily oral tablets. Widely prescribed including by primary care providers.
2. **Long-acting injectables (LAIs)** — monthly or quarterly injections administered in a clinical setting. Used for patients with adherence challenges or treatment-resistant illness.
3. **Clozapine** — reserved exclusively for treatment-resistant schizophrenia. Requires mandatory REMS monitoring with regular blood draws. Represents the most specialized prescribing in this class.

In [3]:
oral_generics = [
    'Quetiapine Fumarate', 'Risperidone', 'Aripiprazole', 'Olanzapine',
    'Ziprasidone Hcl', 'Haloperidol', 'Lurasidone Hcl', 'Cariprazine Hcl',
    'Brexpiprazole', 'Paliperidone', 'Iloperidone', 'Asenapine Maleate',
    'Lumateperone Tosylate', 'Pimavanserin Tartrate', 'Perphenazine',
    'Fluphenazine Hcl', 'Thioridazine Hcl', 'Thiothixene',
    'Trifluoperazine Hcl', 'Chlorpromazine Hcl', 'Loxapine Succinate', 'Pimozide'
]

lais = [
    'Paliperidone Palmitate', 'Aripiprazole Lauroxil', 'Haloperidol Decanoate',
    'Fluphenazine Decanoate', 'Risperidone Microspheres'
]

clozapine = ['Clozapine']

print(f"Oral generics: {len(oral_generics)} drugs")
print(f"LAIs: {len(lais)} drugs")
print(f"Clozapine: {len(clozapine)} drug")

Oral generics: 22 drugs
LAIs: 5 drugs
Clozapine: 1 drug


## Step 2: Build Per-Capita State Tables by Group

Each group is aggregated across all constituent drugs to produce one total claims figure per state, then normalized by Part D enrollment.

In [4]:
def get_group_data(drug_names):
    placeholders = ','.join(['?' for _ in drug_names])
    return pd.read_sql_query(f"""
        SELECT
            p.Prscrbr_Geo_Desc AS state,
            e.BENE_STATE_ABRVTN AS state_abbr,
            SUM(p.Tot_Clms) AS tot_clms,
            SUM(p.Tot_Benes) AS tot_benes,
            ROUND(SUM(p.Tot_Drug_Cst), 0) AS tot_cost,
            e.PRSCRPTN_DRUG_TOT_BENES AS part_d_enrollees,
            ROUND(SUM(p.Tot_Clms) * 100000.0 / e.PRSCRPTN_DRUG_TOT_BENES, 1) AS clms_per_100k
        FROM part_d_geo p
        JOIN state_enrollment e ON p.Prscrbr_Geo_Cd = e.BENE_FIPS_CD
        WHERE p.Prscrbr_Geo_Lvl = 'State'
          AND p.Gnrc_Name IN ({placeholders})
        GROUP BY p.Prscrbr_Geo_Desc, e.BENE_STATE_ABRVTN, e.PRSCRPTN_DRUG_TOT_BENES
        ORDER BY clms_per_100k DESC
    """, conn, params=drug_names)

df_oral = get_group_data(oral_generics)
df_lai  = get_group_data(lais)
df_cloz = get_group_data(clozapine)

print(f"Oral generics  — states with data: {len(df_oral)}")
print(f"LAIs           — states with data: {len(df_lai)}")
print(f"Clozapine      — states with data: {len(df_cloz)}")

Oral generics  — states with data: 54
LAIs           — states with data: 54
Clozapine      — states with data: 54


## Step 3: Top 15 States by Group

Comparing which states rank highest across groups tests the access vs. burden hypothesis. Rural states appearing consistently across all three groups suggests disease burden. Rural states appearing only in the LAI rankings suggests access-driven prescribing.

In [5]:
print("=== Top 15 — Oral Generics (claims per 100k) ===")
print(df_oral[['state', 'clms_per_100k']].head(15).to_string(index=False))

print("\n=== Top 15 — Long-Acting Injectables (claims per 100k) ===")
print(df_lai[['state', 'clms_per_100k']].head(15).to_string(index=False))

print("\n=== Top 15 — Clozapine (claims per 100k) ===")
print(df_cloz[['state', 'clms_per_100k']].head(15).to_string(index=False))

=== Top 15 — Oral Generics (claims per 100k) ===
               state  clms_per_100k
                Guam        96115.2
         Puerto Rico        70840.7
       Massachusetts        68776.9
District of Columbia        68380.9
        Rhode Island        68102.5
        North Dakota        66875.2
            Missouri        66194.8
         Connecticut        64082.4
            Nebraska        64059.8
           Louisiana        59897.9
         Mississippi        59681.0
              Alaska        59648.3
            New York        59490.9
              Kansas        59064.6
           Minnesota        58822.3

=== Top 15 — Long-Acting Injectables (claims per 100k) ===
               state  clms_per_100k
District of Columbia         8652.4
                Guam         6035.4
        Rhode Island         4311.8
            Missouri         4169.8
         Mississippi         4021.4
           Louisiana         3635.8
                Iowa         3572.8
             Alabama       

## Step 4: Geographic Maps by Group

Choropleth maps for each drug group allow visual comparison of whether the rural clustering pattern is consistent across formulation types or specific to LAIs.

In [6]:
def plot_map(df, title, color_scale='Blues'):
    max_val = df['clms_per_100k'].max()
    fig = px.choropleth(
        df,
        locations='state_abbr',
        locationmode='USA-states',
        color='clms_per_100k',
        scope='usa',
        color_continuous_scale=color_scale,
        range_color=[0, round(max_val * 1.05, 1)],
        title=title,
        hover_name='state',
        hover_data={
            'state_abbr': False,
            'tot_clms': True,
            'clms_per_100k': True
        },
        labels={'clms_per_100k': 'Claims per 100k', 'tot_clms': 'Total Claims'}
    )
    fig.update_layout(height=450, coloraxis_colorbar=dict(title='Claims per 100k'))
    fig.show()

plot_map(df_oral, 'Oral Antipsychotics — Claims per 100k Part D Enrollees (2023)', 'Blues')
plot_map(df_lai,  'Long-Acting Injectable Antipsychotics — Claims per 100k Part D Enrollees (2023)', 'Reds')
plot_map(df_cloz, 'Clozapine — Claims per 100k Part D Enrollees (2023)', 'Greens')

## Step 5: Heatmap — Top 20 States Across Drug Groups

This heatmap shows the 20 highest-reporting states for each group side by side. Values are normalized within each group (0-100 scale) so the three columns are visually comparable despite different absolute claim counts.

States that appear dark across all three columns have consistently elevated antipsychotic prescribing regardless of formulation — suggesting disease burden rather than access-driven prescribing. States that are dark only in the LAI column point toward access patterns.

In [7]:
import plotly.graph_objects as go

# Join all three groups on state — keep all states present in any group
heatmap_df = df_oral[['state', 'clms_per_100k']].rename(columns={'clms_per_100k': 'Oral Generics'}).set_index('state')
heatmap_df = heatmap_df.join(
    df_lai[['state', 'clms_per_100k']].rename(columns={'clms_per_100k': 'Long-Acting Injectables'}).set_index('state'),
    how='outer'
)
heatmap_df = heatmap_df.join(
    df_cloz[['state', 'clms_per_100k']].rename(columns={'clms_per_100k': 'Clozapine'}).set_index('state'),
    how='outer'
)

# Normalize each column 0-100 for visual comparability across groups
def normalize(col):
    return (col - col.min()) / (col.max() - col.min()) * 100

heatmap_norm = heatmap_df.apply(normalize)

# Sort rows by average normalized score descending
heatmap_norm['avg'] = heatmap_norm.mean(axis=1)
heatmap_norm = heatmap_norm.sort_values('avg', ascending=True).drop(columns='avg')
heatmap_df = heatmap_df.loc[heatmap_norm.index]

fig = go.Figure(data=go.Heatmap(
    z=heatmap_norm.values,
    x=heatmap_norm.columns.tolist(),
    y=heatmap_norm.index.tolist(),
    colorscale='Blues',
    text=heatmap_df.round(1).values,
    texttemplate='%{text}',
    textfont=dict(size=9),
    hovertemplate='%{y}<br>%{x}<br>Claims per 100k: %{text}<extra></extra>',
    colorbar=dict(title='Normalized Score')
))

fig.update_layout(
    title='Antipsychotic Prescribing by Drug Group — All States (2023)',
    xaxis_title='Drug Group',
    yaxis_title='State',
    height=1100,
    margin=dict(l=150)
)

fig.show()

## Step 6: Spearman Correlation - Testing Geographic Consistency

Spearman correlation measures whether states that rank high in one drug group tend to rank high in another. It is used here instead of Pearson because the claim counts across groups have very different scales and the relationship may not be linear.

Interpretation:
- r > 0.7: strong consistent geographic pattern across groups
- r 0.4-0.7: moderate correlation — groups share some but not all geographic variation
- r < 0.4: weak correlation — groups have distinct geographic patterns

If all three pairwise correlations are strong, the geographic clustering reflects a shared underlying factor (disease burden, demographics, rural access) rather than anything specific to one drug formulation.

In [10]:
from scipy import stats
import plotly.graph_objects as go

# Align all three groups on state index
combined = df_oral[['state', 'clms_per_100k']].rename(columns={'clms_per_100k': 'Oral Generics'}).set_index('state')
combined = combined.join(
    df_lai[['state', 'clms_per_100k']].rename(columns={'clms_per_100k': 'Long-Acting Injectables'}).set_index('state'),
    how='inner'
)
combined = combined.join(
    df_cloz[['state', 'clms_per_100k']].rename(columns={'clms_per_100k': 'Clozapine'}).set_index('state'),
    how='inner'
)

print(f"States included in all three groups: {len(combined)}\n")


# Compute pairwise Spearman correlations
pairs = [
    ('Oral Generics', 'Long-Acting Injectables'),
    ('Oral Generics', 'Clozapine'),
    ('Long-Acting Injectables', 'Clozapine'),
]

results = []
for a, b in pairs:
    r, p = stats.spearmanr(combined[a], combined[b])
    results.append({'Group A': a, 'Group B': b, 'Spearman r': round(r, 3), 'p-value': round(p, 4)})
    print(f"{a} vs {b}: r = {r:.3f}, p = {p:.4f}")

results_df = pd.DataFrame(results)

# Correlation matrix heatmap
cols = ['Oral Generics', 'Long-Acting Injectables', 'Clozapine']
corr_matrix = combined[cols].corr(method='spearman').round(3)

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=cols,
    y=cols,
    colorscale='RdBu',
    zmid=0,
    zmin=-1, zmax=1,
    text=corr_matrix.values,
    texttemplate='%{text}',
    textfont=dict(size=14),
    colorbar=dict(title='Spearman r')
))

fig.update_layout(
    title='Spearman Correlation Matrix — Antipsychotic Prescribing by Drug Group',
    height=450,
    width=550
)

fig.show()
results_df

States included in all three groups: 54

Oral Generics vs Long-Acting Injectables: r = 0.516, p = 0.0001
Oral Generics vs Clozapine: r = 0.451, p = 0.0006
Long-Acting Injectables vs Clozapine: r = 0.319, p = 0.0189


,Group A,Group B,Spearman r,p-value
0,Oral Generics,Long-Acting Injectables,0.516,0.0001
1,Oral Generics,Clozapine,0.451,0.0006
2,Long-Acting Injectables,Clozapine,0.319,0.0189


## Interpretation

All three pairwise correlations are statistically significant (p < 0.05), confirming that geographic variation in antipsychotic prescribing is not random. However, the moderate effect sizes (r = 0.32 to 0.52) indicate that each drug group has its own distinct geographic drivers rather than a single shared factor.

The weakest correlation is between long-acting injectables and clozapine (r = 0.319). These drugs are both used in severe illness, but they cluster differently: LAIs tend toward rural states where patients cannot make frequent clinic visits, while clozapine requires regular blood monitoring and concentrates where specialty psychiatry infrastructure exists. This divergence is the most clinically meaningful signal in the data and motivates the next step - testing whether rurality independently predicts LAI prescribing rates after accounting for overall antipsychotic burden.


## Step 7: Rurality Correlation

The USDA Rural-Urban Continuum Code (RUCC) classifies every U.S. county on a 9-point scale from large metro (1) to most rural (9). A population-weighted mean RUCC score is computed for each state and joined to the prescribing data.

A positive Spearman correlation between weighted RUCC and claims per 100k would confirm that more rural states have higher prescribing rates. If this effect is stronger for LAIs than for oral generics or clozapine, it supports the access-driven hypothesis.

In [11]:
from scipy import stats

# Load state rurality scores
rurality = pd.read_sql_query("SELECT State, weighted_rucc FROM state_rurality", conn)

def add_rurality(df):
    # Join on state abbreviation
    return df.merge(rurality, left_on='state_abbr', right_on='State', how='inner')

oral_r = add_rurality(df_oral)
lai_r  = add_rurality(df_lai)
cloz_r = add_rurality(df_cloz)

# Spearman correlation: rurality vs claims per 100k
results = []
for name, df_r in [('Oral Generics', oral_r), ('Long-Acting Injectables', lai_r), ('Clozapine', cloz_r)]:
    r, p = stats.spearmanr(df_r['weighted_rucc'], df_r['clms_per_100k'])
    results.append({'Drug Group': name, 'Spearman r': round(r, 3), 'p-value': round(p, 4), 'n': len(df_r)})
    print(f"{name}: r = {r:.3f}, p = {p:.4f}, n = {len(df_r)}")

pd.DataFrame(results)

Oral Generics: r = 0.025, p = 0.8584, n = 54
Long-Acting Injectables: r = 0.001, p = 0.9950, n = 54
Clozapine: r = 0.011, p = 0.9385, n = 54


,Drug Group,Spearman r,p-value,n
0,Oral Generics,0.025,0.8584,54
1,Long-Acting Injectables,0.001,0.9950,54
2,Clozapine,0.011,0.9385,54


## Interpretation - Rurality Hypothesis Rejected

The Spearman correlation between state-level rurality (weighted RUCC score) and antipsychotic prescribing rates is effectively zero across all three drug groups (r = 0.025, 0.001, and 0.011 respectively; all p > 0.85). Rurality does not explain the geographic variation observed in this dataset.

Examining the top prescribing states reveals the actual pattern: Massachusetts, District of Columbia, Rhode Island, Connecticut, and New York consistently rank among the highest — states that are among the most urbanized in the country. This directly contradicts the rural isolation hypothesis.

The geographic clustering is more consistent with two alternative explanations:

**Medicaid dual-eligibility:** Antipsychotics are disproportionately prescribed to low-income populations enrolled in both Medicare and Medicaid. States with high dual-eligible rates, concentrated in the Northeast and Mid-Atlantic, would show elevated Part D antipsychotic claims independent of rurality.

**Urban concentrated poverty and serious mental illness:** Conditions like schizophrenia and bipolar disorder are more prevalent in areas with high rates of poverty, homelessness, and social instability - characteristics more associated with dense urban centers than rural states.

The next step tests these hypotheses by correlating state-level antipsychotic prescribing with CMS dual-eligibility rates, which are already available in the state enrollment table loaded during database setup.


## Step 8: Dual-Eligibility Correlation

The CMS enrollment table contains the number of Medicare beneficiaries in each state who are also enrolled in Medicaid (dual-eligibles). Dual-eligibility is a proxy for low-income status. This population has higher rates of serious mental illness, housing instability, and antipsychotic use.

If the dual-eligibility rate explains the geographic prescribing pattern, states where a higher proportion of Medicare beneficiaries are also on Medicaid should show significantly elevated antipsychotic claims per 100k enrollees.

In [14]:
from scipy import stats

# Pull dual-eligibility rate from enrollment table
dual = pd.read_sql_query("""
    SELECT BENE_STATE_ABRVTN AS state_abbr,
           ROUND(100.0 * DUAL_TOT_BENES / TOT_BENES, 2) AS dual_pct
    FROM state_enrollment
    WHERE TOT_BENES > 0
""", conn)

print(f"States with dual-eligibility data: {len(dual)}")
print()
print("Top 10 states by dual-eligibility rate:")
print(dual.sort_values('dual_pct', ascending=False).head(10).to_string(index=False))

# Correlate dual-eligibility rate with claims per 100k for each group
results = []
for name, df_grp in [('Oral Generics', df_oral), ('Long-Acting Injectables', df_lai), ('Clozapine', df_cloz)]:
    merged = df_grp.merge(dual, on='state_abbr', how='inner')
    r, p = stats.spearmanr(merged['dual_pct'], merged['clms_per_100k'])
    results.append({'Drug Group': name, 'Spearman r': round(r, 3), 'p-value': round(p, 4), 'n': len(merged)})
    print(f"{name}: r = {r:.3f}, p = {p:.4f}, n = {len(merged)}")

pd.DataFrame(results)


States with dual-eligibility data: 57

Top 10 states by dual-eligibility rate:
state_abbr  dual_pct
        DC     38.80
        NY     29.45
        LA     27.85
        CT     27.75
        MS     26.81
        ME     25.67
        CA     25.48
        MA     24.93
        NM     24.66
        AR     21.42
Oral Generics: r = 0.113, p = 0.4177, n = 54
Long-Acting Injectables: r = 0.102, p = 0.4615, n = 54
Clozapine: r = -0.076, p = 0.5874, n = 54


,Drug Group,Spearman r,p-value,n
0,Oral Generics,0.113,0.4177,54
1,Long-Acting Injectables,0.102,0.4615,54
2,Clozapine,-0.076,0.5874,54


## Interpretation - Dual-Eligibility Hypothesis Rejected

The Spearman correlation between state-level dual-eligibility rate and antipsychotic prescribing is also effectively zero across all three drug groups (r = 0.113, 0.102, and -0.076 respectively; all p > 0.41). Socioeconomic status as measured by dual-eligibility does not explain the geographic variation at the state level.

Two plausible hypotheses have now been tested and rejected. This outcome is informative rather than inconclusive. It suggests that state-level aggregation is too coarse to detect the underlying signal. A state like Massachusetts has high dual-eligibility concentrated in Boston but very different patterns in rural western areas. Averaging across the entire state masks county-level variation that likely drives the real relationship.

Answering this question properly would require county-level prescribing data linked to county-level socioeconomic indicators. CMS does publish county-level Part D data, and the USDA RUCC file used in Step 7 is already at the county level. A future extension of this analysis could join those datasets directly rather than aggregating to the state.

This notebook demonstrates a complete hypothesis-driven analytical cycle: observation, hypothesis formation, statistical testing, null result, and identification of the methodological limitation preventing a definitive answer.
